# Preprocessing notebooks

## Imports

In [ ]:
from collections import Counter

from datasets import load_dataset
from datasets import Audio, DatasetDict, concatenate_datasets
import torch
import librosa
import numpy as np

SEED = 42
NUM_PROC = 24
SAMPLING_RATE = 16000
CHUNK_DURATION = 0.5
BATCH_SIZE = 32
THRESHOLD_AUGMENTATION = 0.5 # 1-P chance to augment
AUGMENTE_DATASET = True
BALANCE_DATASET = True
CONVERT_TO_SPECTOGRAM = False
CONVERT_TO_MELSPECTOGRAM = True

## Utils

In [ ]:
def display_dataset_labels_count(_dataset: DatasetDict):
    print(f"Labels: {dataset["train"].features["label"]}")
    train_label_count = Counter(_dataset["train"]["label"])
    print(train_label_count)
    val_label_count = Counter(_dataset["val"]["label"])
    print(val_label_count)
    test_label_count = Counter(_dataset["test"]["label"])
    print(test_label_count)

def dataset_splits_info(_dataset: DatasetDict):
    print(f"Size of splits: train={len(_dataset['train'])}, val={len(_dataset['val'])}, test={len(_dataset['test'])}")

## Load base dataset

In [ ]:
# Load dataset
DS = load_dataset("n1coc4cola/maotouying")
DS_train = DS["train"]
print(len(DS_train))

## Shuffle the dataset and select N % of the dataset, create the split, cast to SAMPLING_RATE

In [ ]:
# Take only n% of the dataset
n_instance = 1
DS_train_shuffled = DS_train.shuffle(seed=SEED).select(range(int(n_instance * len(DS_train))))

In [ ]:
train_test = DS_train_shuffled.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
test_val = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="label")
dataset = DatasetDict({
    "train": train_test["train"],
    "val": test_val["train"],
    "test": test_val["test"],
})

In [ ]:
# Cast to 16khz
for split in dataset.keys():
    dataset[split] = dataset[split].cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

## Dataset info

In [ ]:
dataset_splits_info(dataset)

## Split Dataset in chunks of n seconds

In [ ]:
def split_audio_into_chunks(audio_array, chunk_duration=CHUNK_DURATION, sampling_rate=SAMPLING_RATE):
    samples_per_chunk = int(chunk_duration * sampling_rate)
    num_chunks = audio_array.shape[-1] // samples_per_chunk
    # Only split into full chunks, no padding
    chunks = [audio_array[i * samples_per_chunk:(i + 1) * samples_per_chunk]
              for i in range(num_chunks)]

    return chunks

def chunk_audio_batch(batch):
    # Process by batch to allow multi processing
    all_audios = []
    all_sampling_rates = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        audio_array = audio["array"]
        sampling_rate = audio["sampling_rate"]
        audio_array = torch.tensor(audio_array).float()
        chunks = split_audio_into_chunks(audio_array)

        all_audios.extend([chunk.numpy() for chunk in chunks])
        all_sampling_rates.extend([sampling_rate] * len(chunks))
        all_labels.extend([label] * len(chunks))

    return {
        "audio": all_audios,
        "label": all_labels,
    }


chunked_dataset = DatasetDict()
for split in dataset.keys():
    print(f"Chunking split: {split}, original length: {len(dataset[split])}")
    chunked_split = dataset[split].map(
        chunk_audio_batch,
        batched=True,
        num_proc=NUM_PROC,
        batch_size=BATCH_SIZE,
        remove_columns=dataset[split].column_names,
    )
    print(f"Generated split {split} length, {len(chunked_split)}")
    chunked_dataset[split] = chunked_split

In [ ]:
# Test chunked split
from IPython.lib.display import Audio as AudioDisplay
instance = chunked_dataset["train"][0]
print(f"Instance is {chunked_dataset["train"].features["label"].names[instance["label"]]}")
AudioDisplay(instance["audio"], rate=SAMPLING_RATE)

 ## Balance dataset

In [ ]:
display_dataset_labels_count(chunked_dataset)

In [ ]:
if BALANCE_DATASET:
    balanced_dataset = DatasetDict()
    for split in chunked_dataset.keys():
        split_ds = chunked_dataset[split]
        class0 = split_ds.filter(lambda x: x["label"] == 0, num_proc=NUM_PROC)
        class1 = split_ds.filter(lambda x: x["label"] == 1, num_proc=NUM_PROC)

        n = min(len(class0), len(class1))
        class0_ds = class0.select(range(n))
        class1_ds = class1.select(range(n))

        balanced_dataset[split] = concatenate_datasets([class0_ds, class1_ds]).shuffle(seed=SEED)
else:
    balanced_dataset = chunked_dataset
dataset_splits_info(balanced_dataset)

In [ ]:
display_dataset_labels_count(balanced_dataset)

## Data Augmentation

In [ ]:
from audiomentations import AddGaussianNoise, AddGaussianSNR, AirAbsorption
from torch_audiomentations import Compose, Gain, PolarityInversion

transform_gain_polarity = Compose(
    transforms=[
        Gain(
            min_gain_in_db=-15.0,
            max_gain_in_db=5.0,
            p=0.5,
        ),
        PolarityInversion(p=0.5)
    ]
)

transform_gaussian_noise = AddGaussianNoise(
    min_amplitude=0.001,
    max_amplitude=0.015,
    p=1.0
)

transform_gaussian_snr = AddGaussianSNR(
    min_snr_db=5,
    max_snr_db=20,
    p=1.0
)

transform_airabsorption = AirAbsorption(
    min_distance=10.0,
    max_distance=50.0,
    p=1.0,
)

transforms = [
    transform_gain_polarity,
    transform_gaussian_noise,
    transform_gaussian_snr,
    transform_airabsorption,
]

In [ ]:
def augmente_batch(batch):
    rng = np.random.default_rng()
    all_audios, all_labels = [], []

    for audio, label in zip(batch["audio"], batch["label"]):
        all_audios.append(audio)
        all_labels.append(label)

        if rng.random() > THRESHOLD_AUGMENTATION:
            transform = rng.choice(transforms)
            data = np.array(audio).astype(np.float32)
            all_audios.append(data)
            all_labels.append(label)

            # Handle torch_audiomentations transforms
            if isinstance(transform, Compose) or (hasattr(transform, '__module__') and 'torch_audiomentations' in str(getattr(transform, '__module__', ''))):
                tensor = torch.tensor(data).unsqueeze(0).unsqueeze(0)
                result = transform(tensor, sample_rate=SAMPLING_RATE)
                data = result.squeeze().numpy() if isinstance(result, torch.Tensor) else result.samples.squeeze().numpy()
            else:
                data = transform(data, sample_rate=SAMPLING_RATE)

            all_audios.append(data)
            all_labels.append(label)

    return {"audio": all_audios, "label": all_labels}

if AUGMENTE_DATASET:

    print(f"Augmenting train split, length before: {len(balanced_dataset["train"])}")
    augmented_split = balanced_dataset["train"].map(
        augmente_batch,
        batched=True,
        num_proc=NUM_PROC,
        batch_size=BATCH_SIZE,
        remove_columns=balanced_dataset["train"].column_names,
    )
    augmented_split = augmented_split.shuffle(seed=SEED+1)
    print(f"Length after: {len(augmented_split)}")

    augmented_dataset = DatasetDict({
        "train": augmented_split,
        "val": balanced_dataset["val"],
        "test": balanced_dataset["test"]
    })
else:
    augmented_dataset = balanced_dataset


## Convert to spectograms

In [ ]:
if CONVERT_TO_SPECTOGRAM and CONVERT_TO_MELSPECTOGRAM:
    print("CAN'T CONVERT TO SPECRTOGRAM AND MELSPECTOGRAM")

In [ ]:
# -----------------------------
# Audio transforms for linear spectrogram
# -----------------------------
def convert_to_linear_spectrogram(batch):
    all_linear_db = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        data = np.array(audio)
        # Compute STFT
        stft = librosa.stft(data, n_fft=2048, hop_length=256)

        # Compute magnitude
        magnitude = np.abs(stft)

        # Convert to dB
        linear_db = librosa.amplitude_to_db(magnitude, ref=np.max)
        all_linear_db.append(torch.tensor(linear_db))
        all_labels.append(torch.tensor(label))

    return {
        # Convert to torch tensors
        "audio": all_linear_db,
        "label": all_labels,
    }



if CONVERT_TO_SPECTOGRAM and not CONVERT_TO_MELSPECTOGRAM:
    spectrogram_dataset = DatasetDict()
    for split in augmented_dataset.keys():
        print(f"Converting to spectrogram split: {split}")
        spectrogram_split = augmented_dataset[split].map(
            convert_to_linear_spectrogram,
            batched=True,
            num_proc=NUM_PROC,
            batch_size=BATCH_SIZE,
            remove_columns=augmented_dataset[split].column_names,
        )
        spectrogram_dataset[split] = spectrogram_split
else:
    spectrogram_dataset = augmented_dataset


In [ ]:
# -----------------------------
# Audio transforms for linear spectrogram
# -----------------------------
def convert_to_mel_spectrogram(batch):
    all_linear_db = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        data = np.array(audio)

        mel = librosa.feature.melspectrogram(
            y=data,
            sr=SAMPLING_RATE,
            n_fft=1025,
            hop_length=256,
            n_mels=128,
            fmin=20,
            fmax=8000,
            power=2.0
        )
         # Convert to log scale (dB)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        # Normalize
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        # Convert to torch tensor: [1, n_mels, time]
        mel_db = torch.tensor(mel_db).unsqueeze(0)

        all_linear_db.append(mel_db)
        all_labels.append(torch.tensor(label))

    return {
        # Convert to torch tensors
        "audio": all_linear_db,
        "label": all_labels,
    }



if CONVERT_TO_MELSPECTOGRAM and not CONVERT_TO_SPECTOGRAM:
    spectrogram_dataset = DatasetDict()
    for split in augmented_dataset.keys():
        print(f"Converting to melspectrogram split: {split}")
        melspectrogram_split = augmented_dataset[split].map(
            convert_to_mel_spectrogram,
            batched=True,
            num_proc=NUM_PROC-4,
            batch_size=BATCH_SIZE,
            remove_columns=augmented_dataset[split].column_names,
        )
        spectrogram_dataset[split] = melspectrogram_split
else:
    spectrogram_dataset = augmented_dataset


In [ ]:
import matplotlib.pyplot as plt
# Debug spectogram
if CONVERT_TO_SPECTOGRAM:
    instance = spectrogram_dataset["train"][1]
    array = np.array(instance["audio"])
    plt.figure(figsize=(10, 10))
    librosa.display.specshow(array, sr=SAMPLING_RATE, x_axis="time", y_axis="log")
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"Spectrogram of {spectrogram_dataset["train"].features["label"].names[instance["label"]]}")
    plt.show()


4## Save to disk the dataset

In [ ]:
dataset_name = f"ds_{str(n_instance).replace(".", "-")}_{"specto" if CONVERT_TO_SPECTOGRAM else "melspecto" if CONVERT_TO_MELSPECTOGRAM else "raw"}_{("aug_"+str(THRESHOLD_AUGMENTATION).replace(".", "-")) if AUGMENTE_DATASET else "noaug"}_{"balanced" if BALANCE_DATASET else "not_balanced"}_chunked.hf"
dataset_name

In [ ]:
spectrogram_dataset.save_to_disk(f"./{dataset_name}")

In [ ]:
from huggingface_hub import login
login("")

In [ ]:
spectrogram_dataset.push_to_hub("Hibou-Foundation/ds_0-001_specto_aug_0-8_balanced_chunked")